# INN Hotels Project

## Context

A significant number of hotel bookings are called-off due to cancellations or no-shows. The typical reasons for cancellations include change of plans, scheduling conflicts, etc. This is often made easier by the option to do so free of charge or preferably at a low cost which is beneficial to hotel guests but it is a less desirable and possibly revenue-diminishing factor for hotels to deal with. Such losses are particularly high on last-minute cancellations.

The new technologies involving online booking channels have dramatically changed customers’ booking possibilities and behavior. This adds a further dimension to the challenge of how hotels handle cancellations, which are no longer limited to traditional booking and guest characteristics.

The cancellation of bookings impact a hotel on various fronts:
* Loss of resources (revenue) when the hotel cannot resell the room.
* Additional costs of distribution channels by increasing commissions or paying for publicity to help sell these rooms.
* Lowering prices last minute, so the hotel can resell a room, resulting in reducing the profit margin.
* Human resources to make arrangements for the guests.

## Objective
The increasing number of cancellations calls for a Machine Learning based solution that can help in predicting which booking is likely to be canceled. INN Hotels Group has a chain of hotels in Portugal, they are facing problems with the high number of booking cancellations and have reached out to your firm for data-driven solutions. You as a data scientist have to analyze the data provided to find which factors have a high influence on booking cancellations, build a predictive model that can predict which booking is going to be canceled in advance, and help in formulating profitable policies for cancellations and refunds.

## Data Description
The data contains the different attributes of customers' booking details. The detailed data dictionary is given below.


**Data Dictionary**

* Booking_ID: unique identifier of each booking
* no_of_adults: Number of adults
* no_of_children: Number of Children
* no_of_weekend_nights: Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel
* no_of_week_nights: Number of week nights (Monday to Friday) the guest stayed or booked to stay at the hotel
* type_of_meal_plan: Type of meal plan booked by the customer:
    * Not Selected – No meal plan selected
    * Meal Plan 1 – Breakfast
    * Meal Plan 2 – Half board (breakfast and one other meal)
    * Meal Plan 3 – Full board (breakfast, lunch, and dinner)
* required_car_parking_space: Does the customer require a car parking space? (0 - No, 1- Yes)
* room_type_reserved: Type of room reserved by the customer. The values are ciphered (encoded) by INN Hotels.
* lead_time: Number of days between the date of booking and the arrival date
* arrival_year: Year of arrival date
* arrival_month: Month of arrival date
* arrival_date: Date of the month
* market_segment_type: Market segment designation.
* repeated_guest: Is the customer a repeated guest? (0 - No, 1- Yes)
* no_of_previous_cancellations: Number of previous bookings that were canceled by the customer prior to the current booking
* no_of_previous_bookings_not_canceled: Number of previous bookings not canceled by the customer prior to the current booking
* avg_price_per_room: Average price per day of the reservation; prices of the rooms are dynamic. (in euros)
* no_of_special_requests: Total number of special requests made by the customer (e.g. high floor, view from the room, etc)
* booking_status: Flag indicating if the booking was canceled or not.

## Importing necessary libraries and data

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/My Drive/INNHotelsGroup.csv')

ModuleNotFoundError: No module named 'google'

## Data Overview

- Observations
- Sanity checks

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info

In [ ]:
statistical_summary = df.describe()
print(statistical_summary)

In [ ]:
duplicates = df[df.duplicated()]
print("Duplicate rows in the DataFrame:")
print(duplicates)

In [ ]:
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

## Exploratory Data Analysis (EDA)

- EDA is an important part of any project involving data.
- It is important to investigate and understand the data better before building a model with it.
- A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.
- A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Leading Questions**:
1. What are the busiest months in the hotel?
2. Which market segment do most of the guests come from?
3. Hotel rates are dynamic and change according to demand and customer demographics. What are the differences in room prices in different market segments?
4. What percentage of bookings are canceled?
5. Repeating guests are the guests who stay in the hotel often and are important to brand equity. What percentage of repeating guests cancel?
6. Many guests have special requirements when booking a hotel room. Do these requirements affect booking cancellation?

### Univariate Analysis

In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,
        sharex=True,
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )

In [ ]:
def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )
        else:
            label = p.get_height()

        x = p.get_x() + p.get_width() / 2
        y = p.get_height()

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.show()


### Observations on lead time

In [ ]:
histogram_boxplot(df, "lead_time")

### Observations on average price per room

In [ ]:
histogram_boxplot(df,'avg_price_per_room')

In [ ]:
df[df["avg_price_per_room"] == 0]

In [ ]:
df.loc[df["avg_price_per_room"] == 0, "market_segment_type"].value_counts()

In [ ]:
Q1 = df["avg_price_per_room"].quantile(0.25)

In [ ]:
Q3 = df["avg_price_per_room"].quantile(0.75)

In [ ]:
IQR = Q3 - Q1

In [ ]:
Upper_Whisker = Q3 + 1.5 * IQR
Upper_Whisker

In [ ]:
df.loc[df["avg_price_per_room"] >= 500, "avg_price_per_room"] = Upper_Whisker

### Observations on number of previous booking cancellations

In [ ]:
histogram_boxplot(df,'no_of_previous_cancellations')

### Observations on number of previous booking not canceled

In [ ]:
histogram_boxplot(df,'no_of_previous_bookings_not_canceled')

In [ ]:
def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )
        else:
            label = p.get_height()

        x = p.get_x() + p.get_width() / 2
        y = p.get_height()
        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.show()

In [ ]:
labeled_barplot(df, "no_of_adults", perc=True)

In [ ]:
labeled_barplot(df,'no_of_children')

In [ ]:
df["no_of_children"] = df["no_of_children"].replace([9, 10], 3)

### Observations on number of week nights

In [ ]:
labeled_barplot(df,'no_of_week_nights')

In [ ]:
labeled_barplot(df,'no_of_weekend_nights')

In [ ]:
labeled_barplot(df,'required_car_parking_space')

In [ ]:
labeled_barplot(df,'type_of_meal_plan')

In [ ]:
labeled_barplot(df,'room_type_reserved')

In [ ]:
labeled_barplot(df,'arrival_month')

In [ ]:
labeled_barplot(df,'market_segment_type')

In [ ]:
labeled_barplot(df,'no_of_special_requests')

In [ ]:
labeled_barplot(df,'booking_status')

In [ ]:
df["booking_status"] = df["booking_status"].apply(
    lambda x: 1 if x == "Canceled" else 0
)

# Bivariate Analysis

In [ ]:
cols_list = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(12, 7))
sns.heatmap(
    df[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.show()

In [ ]:
def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
        stat="density",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
        stat="density",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df, x="market_segment_type", y="avg_price_per_room", palette="gist_rainbow"
)
plt.show()

In [ ]:
stacked_barplot(df, "market_segment_type", "booking_status")

In [ ]:
stacked_barplot(df, "no_of_special_requests", "booking_status")

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot('df, "no_of_special_requests", "avg_price_per_room"')
plt.show()

In [ ]:
distribution_plot_wrt_target(df, "avg_price_per_room", "booking_status")

In [ ]:
distribution_plot_wrt_target(df, "avg_price_per_room", "booking_status")

In [ ]:
distribution_plot_wrt_target(df, "lead_time", "booking_status") ## Complete the code to find distribution of lead time wrt booking status

Generally people travel with their spouse and children for vacations or other activities. Let's create a new dataframe of the customers who traveled with their families and analyze the impact on booking status.

In [ ]:
family_data = df[(df["no_of_children"] >= 0) & (df["no_of_adults"] > 1)]
family_data.shape

In [ ]:
family_data["no_of_family_members"] = (
    family_data["no_of_adults"] + family_data["no_of_children"]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data_to_plot = family_data.groupby(['no_of_family_members', 'booking_status']).size().unstack(fill_value=0)

data_to_plot.plot(kind='bar', stacked=True)

plt.xlabel('Number of Family Members')
plt.ylabel('Count')
plt.title('Stacked Barplot of Family Members vs Booking Status')
plt.legend(title='Booking Status')

plt.show()


**Let's do a similar analysis for the customer who stay for at least a day at the hotel.**


In [ ]:
stay_data = df[(df["no_of_week_nights"] > 0) & (df["no_of_weekend_nights"] > 0)]
stay_data.shape

In [ ]:
stay_data["total_days"] = (
    stay_data["no_of_week_nights"] + stay_data["no_of_weekend_nights"]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data_to_plot = stay_data.groupby(['total_days', 'booking_status']).size().unstack(fill_value=0)

data_to_plot.plot(kind='bar', stacked=True)

plt.xlabel('Total Days')
plt.ylabel('Count')
plt.title('Stacked Barplot of Total Days vs Booking Status')
plt.legend(title='Booking Status')

plt.show()


Repeating guests are the guests who stay in the hotel often and are important to brand equity. Let's see what percentage of repeating guests cancel?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data_to_plot = stay_data.groupby(['repeated_guest', 'booking_status']).size().unstack(fill_value=0)

data_to_plot.plot(kind='bar', stacked=True)

plt.xlabel('Repeated Guest')
plt.ylabel('Count')
plt.title('Stacked Barplot of Repeated Guests vs Booking Status')
plt.legend(title='Booking Status')

plt.show()


**Let's find out what are the busiest months in the hotel.**

In [ ]:
monthly_data = df.groupby(["arrival_month"])["booking_status"].count()

monthly_data = pd.DataFrame(
    {"Month": list(monthly_data.index), "Guests": list(monthly_data.values)}
)

plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly_data, x="Month", y="Guests")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data_to_plot = df.groupby(['arrival_month', 'booking_status']).size().unstack(fill_value=0)

data_to_plot.plot(kind='bar', stacked=True, figsize=(10, 5))

plt.xlabel('Arrival Month')
plt.ylabel('Count')
plt.title('Stacked Barplot of Arrival Month vs Booking Status')
plt.legend(title='Booking Status')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='arrival_month', y='avg_price_per_room')

plt.xlabel('Arrival Month')
plt.ylabel('Average Price per Room')
plt.title('Lineplot of Average Price per Room vs Arrival Month')

plt.show()



### Outlier Check

- Let's check for outliers in the data.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(df[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

## Model Building

### Logistic Regression (with statsmodels library)

#### Data Preparation for modeling (Logistic Regression)

In [ ]:
import statsmodels.api as sm
import pandas as pd
from sklearn.model_selection import train_test_split

X = df.drop(["booking_status"], axis=1)
Y = df["booking_status"]

X = sm.add_constant(X)

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)


In [ ]:
print("Shape of Training set : ", X_train.shape)
print("Shape of test set : ", X_test.shape)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

#### Building Logistic Regression Model

In [ ]:
X = df.drop(["booking_status"], axis=1)
Y = df["booking_status"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)


In [ ]:
X = df.drop(["booking_status", "Booking_ID"], axis=1)


In [ ]:
X = pd.get_dummies(X, drop_first=True)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

X = df.drop(["booking_status", "Booking_ID"], axis=1)
X = pd.get_dummies(X, drop_first=True)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, df["booking_status"], test_size=0.3, random_state=1)


In [ ]:
X_train = X_train.to_numpy()

In [ ]:
y_train = np.asarray(y_train)
X_train = np.asarray(X_train)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

Y = label_encoder.fit_transform(df['booking_status'])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

logit = sm.Logit(y_train, X_train)

lg = logit.fit()

print(lg.summary())


In [ ]:
logit = sm.Logit(y_train, X_train)

In [ ]:
print("Training performance:")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def model_performance_classification_statsmodels(model, X, y_true):

    y_pred_prob = model.predict(X)

    y_pred = [1 if prob > 0.5 else 0 for prob in y_pred_prob]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

print("Training performance:")
model_performance_classification_statsmodels(lg, X_train, y_train)


In [ ]:
!pip install statsmodels

#### Multicollinearity

In [ ]:
def checking_vif(predictors):
    vif = pd.DataFrame()
    vif["feature"] = predictors.columns

    vif["VIF"] = [
        variance_inflation_factor(predictors.values, i)
        for i in range(len(predictors.columns))
    ]
    return vif

In [ ]:
checking_vif(X_train)

#### Dropping high p-value variables

- We will drop the predictor variables having a p-value greater than 0.05 as they do not significantly impact the target variable.
- But sometimes p-values change after dropping a variable. So, we'll not drop all variables at once.
- Instead, we will do the following:
    - Build a model, check the p-values of the variables, and drop the column with the highest p-value.
    - Create a new model without the dropped feature, check the p-values of the variables, and drop the column with the highest p-value.
    - Repeat the above two steps till there are no columns with p-value > 0.05.

The above process can also be done manually by picking one variable at a time that has a high p-value, dropping it, and building a model again. But that might be a little tedious and using a loop will be more efficient.

In [ ]:
cols = X_train.columns.tolist()

max_p_value = 1

while len(cols) > 0:

    x_train_aux = X_train[cols]

    model = sm.Logit(y_train, x_train_aux).fit(disp=False)

    p_values = model.pvalues
    max_p_value = max(p_values)

    feature_with_p_max = p_values.idxmax()

    if max_p_value > 0.05:
        cols.remove(feature_with_p_max)
    else:
        break

selected_features = cols
print(selected_features)

In [ ]:
X_train1 = X_train[selected_features]
X_test1 = X_test[selected_features]

In [ ]:
import statsmodels.api as sm

logit1 = sm.Logit(y_train, X_train1)


lg1 = logit1.fit()

print(lg1.summary())


In [ ]:
print("Training performance:")
model_performance_classification_statsmodels(lg1, X_train1, y_train)


In [ ]:
odds = np.exp(lg1.params)

perc_change_odds = (np.exp(lg1.params) - 1) * 100

pd.set_option("display.max_columns", None)

pd.DataFrame({"Odds": odds, "Change_odd%": perc_change_odds}, index=X_train1.columns).T

In [ ]:
!pip install statsmodels
import statsmodels.api as sm

In [ ]:
def confusion_matrix_statsmodels(model, X, y):

    y_pred_prob = model.predict(X).values

    y_pred = (y_pred_prob >= 0.5).astype(int)

    confusion_matrix = pd.crosstab(y, y_pred, rownames=['Actual'], colnames=['Predicted'])

    accuracy = (confusion_matrix.iloc[0, 0] + confusion_matrix.iloc[1, 1]) / len(y)

    precision = confusion_matrix.iloc[1, 1] / (confusion_matrix.iloc[1, 1] + confusion_matrix.iloc[0, 1])

    recall = confusion_matrix.iloc[1, 1] / (confusion_matrix.iloc[1, 1] + confusion_matrix.iloc[1, 0])

    f1_score = 2 * (precision * recall) / (precision + recall)

    print("Confusion Matrix:")
    print(confusion_matrix)
    print("\nAccuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1_score)

In [ ]:
confusion_matrix_statsmodels(lg1, X_train1, y_train)

In [ ]:
print("Training performance:")

log_reg_model_train_perf


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

#### ROC-AUC
* ROC-AUC on training set

In [ ]:
logit_roc_auc_train = roc_auc_score(y_train, lg1.predict(X_train1))
fpr, tpr, thresholds = roc_curve(y_train, lg1.predict(X_train1))
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label="Logistic Regression (area = %0.2f)" % logit_roc_auc_train)
plt.plot([0, 1], [0, 1], "r--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.01])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver operating characteristic")
plt.legend(loc="lower right")
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_train, lg1.predict(X_train1))

optimal_idx = np.argmax(tpr - fpr)
optimal_threshold_auc_roc = thresholds[optimal_idx]
print(optimal_threshold_auc_roc)

In [ ]:
confusion_matrix_statsmodels(lg1, X_train1, y_train)

In [ ]:
import statsmodels.api as sm

In [ ]:
log_reg_model_train_perf_threshold_auc_roc = model_performance_classification_statsmodels(
    lg1, X_train1, y_train
)

In [ ]:
print("Training performance:")
log_reg_model_train_perf_threshold_auc_roc

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.metrics import precision_recall_curve

In [ ]:
from sklearn.metrics import precision_recall_curve

y_scores = lg1.predict(X_train1)
prec, rec, tre = precision_recall_curve(y_train, y_scores)


def plot_prec_recall_vs_tresh(precisions, recalls, thresholds):
    plt.plot(thresholds, precisions[:-1], "b--", label="precision")
    plt.plot(thresholds, recalls[:-1], "g--", label="recall")
    plt.xlabel("Threshold")
    plt.legend(loc="upper left")
    plt.ylim([0, 1])


plt.figure(figsize=(10, 7))
plot_prec_recall_vs_tresh(prec, rec, tre)
plt.show()

In [ ]:
optimal_threshold_curve = 0.42

#### Checking model performance on training set

In [ ]:
!pip install scikit-learn

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_train, lg1.predict(X_train1) > optimal_threshold_curve)

print(cm)

In [ ]:
!pip install scikit-learn

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_train, lg1.predict(X_train1) > optimal_threshold_curve)

print(cm)

In [ ]:
from yellowbrick.classifier import ClassificationReport

In [ ]:
from sklearn.linear_model import LogisticRegression
from yellowbrick.classifier import ClassificationReport

In [ ]:
print(type(lg1))

In [ ]:
from sklearn.linear_model import LogisticRegression
from yellowbrick.classifier import ClassificationReport

In [ ]:
lg1 = LogisticRegression()

In [ ]:
lg1.fit(X_train, y_train)

In [ ]:
visualizer = ClassificationReport(lg1, classes=["0", "1"])
visualizer.show()

#### Performance on the test set

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lg1 = LogisticRegression()

In [ ]:
lg1.fit(X_train, y_train)

In [ ]:
!pip install statsmodels

In [ ]:
import statsmodels.api as sm

In [ ]:
import statsmodels.api as sm
import pandas as pd

In [ ]:
print(missing_features)

In [ ]:
print(X_test1.columns[X_test1.columns.isin(missing_features)])

In [ ]:
missing_features = list(set(lg1.feature_names_in_) - set(X_test1.columns))

if missing_features:
    X_test1[missing_features] = 0

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
print(X_test1.columns)

In [ ]:
print(lg1.feature_names_in_)

In [ ]:
if not np.array_equal(X_test1.columns, lg1.feature_names_in_):
    X_test1 = X_test1[lg1.feature_names_in_]

In [ ]:
lg1 = sm.Logit(y_train, X_train)

In [ ]:
lg1.fit()

In [ ]:
!pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
print(X_test1.shape)
print(y_test.shape)

In [ ]:
print(dir(lg1))

In [ ]:
print(type(lg1))

In [ ]:
!pip install --upgrade statsmodels
import statsmodels.api as sm
help(sm.Logit)

In [ ]:
print(type(lg1))

In [ ]:
print(lg1)

In [ ]:
if hasattr(lg1, "params"):
    num_params = len(lg1.params)
    print(num_params)
else:
    print("The 'Logit' object does not have a 'params' attribute.")

In [ ]:
!pip install scikit-learn

from sklearn.linear_model import LogisticRegression

lg1 = LogisticRegression()

if hasattr(lg1, 'params'):
    print(len(lg1.params))

else:
    print("The 'params' attribute is not available in this version of scikit-learn.")

In [ ]:
!pip install scikit-learn==0.24.2

from sklearn.linear_model import LogisticRegression

lg1 = LogisticRegression(random_state=1)

num_params = len(lg1.get_params())

print(num_params)

In [ ]:
!pip install scikit-learn

from sklearn.linear_model import LogisticRegression

lg1 = LogisticRegression()

num_params = len(lg1.get_params())

print(num_params)

In [ ]:
num_params = len(lg1.get_params())

In [ ]:
!ls -l INNHotelsGroup.csv

data_path = "INNHotelsGroup.csv"


data = pd.read_csv('/content/drive/My Drive/INNHotelsGroup.csv')

In [ ]:
df = pd.read_csv('/content/drive/My Drive/INNHotelsGroup.csv')

In [ ]:
print(df.columns)


In [ ]:
print(df.columns)

#### Building Logistic Regression Model

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [ ]:
X = data.drop(["booking_status"], axis=1)
Y = data["booking_status"]

X = pd.get_dummies(X, drop_first=True)

label_encoder = LabelEncoder()
Y = label_encoder.fit_transform(Y)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)


In [ ]:
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)


In [ ]:
logit_model = sm.Logit(y_train, X_train_sm)

result = logit_model.fit()

In [ ]:
print(result.summary())


In [ ]:
y_pred_prob = result.predict(X_test_sm)

y_pred = [1 if prob > 0.5 else 0 for prob in y_pred_prob]


## Checking Multicollinearity

- In order to make statistical inferences from a logistic regression model, it is important to ensure that there is no multicollinearity present in the data.

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split


In [ ]:
X = df.drop(['booking_status'], axis=1)
Y = df['booking_status']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

In [ ]:
X_train_const = sm.add_constant(X_train)

vif_data = pd.DataFrame()
vif_data['Feature'] = X_train_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_train_const.values, i) for i in range(X_train_const.shape[1])]


In [ ]:
print(vif_data)


## Model performance evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt


In [ ]:
y_pred_prob = model.predict(X_test)

threshold = 0.5
y_pred = [1 if prob > threshold else 0 for prob in y_pred_prob]


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.2f}")

recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.2f}")

f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.2f}")

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()


## Final Model Summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
print(f"Final Model Accuracy: {final_accuracy:.2f}")
print(f"Final Model Precision: {final_precision:.2f}")
print(f"Final Model Recall: {final_recall:.2f}")
print(f"Final Model F1 Score: {final_f1:.2f}")
print(f"Final Model ROC-AUC Score: {final_roc_auc:.2f}")


In [ ]:
conf_matrix = confusion_matrix(y_test, final_y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
class_report = classification_report(y_test, final_y_pred)
print("Classification Report:")
print(class_report)


In [ ]:
feature_imp = final_model.feature_importances_
sns.barplot(x=feature_imp, y=X.columns)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


## Decision Tree

In [ ]:
X = pd.get_dummies(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)


In [ ]:
import pandas as pd

print("Shape of Training set : ", X_train.shape)
print("Shape of test set : ", X_test.shape)
print("Percentage of classes in training set:")
print(pd.Series(y_train).value_counts(normalize=True))
print("Percentage of classes in test set:")
print(pd.Series(y_test).value_counts(normalize=True))

In [ ]:
print(type(y_train))
print(type(y_test))

In [ ]:
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
print("Shape of Training set : ", X_train.shape)
print("Shape of test set : ", X_test.shape)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

#### Creating functions to calculate metrics and confusion matrix

In [ ]:
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

#### Building Decision Tree Model

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
!wget https://raw.githubusercontent.com/StillWork/data/master/iris.csv

In [ ]:
!ls -l iris.csv

In [ ]:
!wget https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv

In [ ]:
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

#### Checking model performance on training set

In [ ]:
confusion_matrix_sklearn(model, X_train, y_train)


In [ ]:
decision_tree_perf_train = model_performance_classification_sklearn(
    model, X_train, y_train
)
decision_tree_perf_train

#### Checking model performance on test set

In [ ]:
confusion_matrix_sklearn(model, X_test, y_test)


In [ ]:
decision_tree_perf_test = model_performance_classification_sklearn(model, X_test, y_test)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

feature_names = list(X_train.columns)
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()


Pruning ther Tree

In [ ]:
estimator = DecisionTreeClassifier(random_state=1, class_weight="balanced")

parameters = {
    "max_depth": np.arange(2, 7, 2),
    "max_leaf_nodes": [50, 75, 150, 250],
    "min_samples_split": [10, 30, 50, 70],
}

acc_scorer = make_scorer(f1_score)

grid_obj = GridSearchCV(estimator, parameters, scoring=acc_scorer, cv=5)
grid_obj = grid_obj.fit(X_train, y_train)

estimator = grid_obj.best_estimator_

estimator.fit(X_train, y_train)

In [ ]:
confusion_matrix_sklearn(model, X_train, y_train)


In [ ]:
decision_tree_tune_perf_train = model_performance_classification_sklearn('model, X_train, y_train')
decision_tree_tune_perf_train

#### Checking performance on test set

In [ ]:
confusion_matrix_sklearn('model, X_test, y_test')

In [ ]:
decision_tree_tune_perf_test = model_performance_classification_sklearn('model, X_test, y_test')
decision_tree_tune_perf_test

Visualizing the Decision Tree

In [ ]:
plt.figure(figsize=(20, 10))
out = tree.plot_tree(
    estimator,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)

for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:

print(tree.export_text(estimator, feature_names=feature_names, show_weights=True))

In [ ]:
importances = estimator.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

In [ ]:
clf = DecisionTreeClassifier(random_state=1, class_weight="balanced")
path = clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = abs(path.ccp_alphas), path.impurities

In [ ]:
pd.DataFrame(path)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")
plt.show()

In [ ]:
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(
        random_state=1, ccp_alpha=ccp_alpha, class_weight="balanced"
    )
    clf.'X_train, y_train'
    clfs.append(clf)
print(
    "Number of nodes in the last tree is: {} with ccp_alpha: {}".format(
        clfs[-1].tree_.node_count, ccp_alphas[-1]
    )
)

In [ ]:
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

node_counts = [clf.tree_.node_count for clf in clfs]
depth = [clf.tree_.max_depth for clf in clfs]
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(ccp_alphas, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_xlabel("alpha")
ax[0].set_ylabel("number of nodes")
ax[0].set_title("Number of nodes vs alpha")
ax[1].plot(ccp_alphas, depth, marker="o", drawstyle="steps-post")
ax[1].set_xlabel("alpha")
ax[1].set_ylabel("depth of tree")
ax[1].set_title("Depth vs alpha")
fig.tight_layout()

F1 Score vs alpha for training and testing sets

In [ ]:
f1_train = []
for clf in clfs:
    pred_train = clf.predict(X_train)
    values_train = f1_score(y_train, pred_train)
    f1_train.append(values_train)

f1_test = []
for clf in clfs:
    pred_test = clf.predict(X_test)
    values_test = f1_score(y_test, pred_test)
    f1_test.append(values_test)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlabel("alpha")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score vs alpha for training and testing sets")
ax.plot(ccp_alphas, f1_train, marker="o", label="train", drawstyle="steps-post")
ax.plot(ccp_alphas, f1_test, marker="o", label="test", drawstyle="steps-post")
ax.legend()
plt.show()

In [ ]:
index_best_model = np.argmax(f1_test)
best_model = clfs[index_best_model]
print(best_model)

Checking performance on training set

In [ ]:
confusion_matrix_sklearn(best_model, X_train, y_train)

In [ ]:
decision_tree_post_perf_train = model_performance_classification_sklearn(
    best_model, X_train, y_train
)
decision_tree_post_perf_train

Checking performance on test set

In [ ]:
confusion_matrix_sklearn(best_model, X_test, y_test)


In [ ]:
decision_tree_post_test = model_performance_classification_sklearn(best_model, X_test, y_test)'
decision_tree_post_test

In [ ]:
plt.figure(figsize=(20, 10))

out = tree.plot_tree(
    best_model,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
importances = best_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

Comparing Decision Tree models

In [ ]:
models_train_comp_df = pd.concat(
    [
        decision_tree_perf_train.T,
        decision_tree_tune_perf_train.T,
        decision_tree_post_perf_train.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Decision Tree sklearn",
    "Decision Tree (Pre-Pruning)",
    "Decision Tree (Post-Pruning)",
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
compare_models(original_model_perf, decision_tree_post_test)


## Do we need to prune the tree?

Pruning was necessary and beneficial for the decision tree model to prevent overfitting and improve its generalizability to unseen data.

## Model Performance Comparison and Conclusions

**Model Building - Logistic Regression**

- The logistic regression model indicated significant predictors like lead time, special requests, and room type, with their coefficients providing insights into their impact on cancellation likelihood.
The model's assumptions, such as linearity in the log-odds and lack of multicollinearity, were addressed during pre-processing and feature selection.

**Model Improvement - Logistic Regression**

- Adjusting the classification threshold based on ROC curve analysis improved model sensitivity and specificity, better aligning with business objectives by possibly reducing false negatives (undetected cancellations).

**Model Building - Decision Tree**

- The decision tree model offered a different approach, providing a clear set of decision rules and highlighting the most influential factors leading to cancellations.

- Initial performance indicated areas for improvement, primarily in reducing overfitting and complexity.

**Model Improvement - Decision Tree**

- Pruning the decision tree using cost complexity pruning (ccp_alpha) improved the model by reducing complexity and overfitting, making it more generalizable. The decision rules and feature importance derived from the pruned tree offered actionable insights, with lead time and special requests again being significant.

## Actionable Insights and Recommendations

**Exploratory Data Analysis**
- The primary issue INN Hotels face is the high rate of booking cancellations, impacting revenue, resource allocation, and operational efficiency. The objective is to predict which bookings are likely to be canceled to mitigate these challenges.

**Univariate Analysis**
- Booking Status: The target variable shows a distribution between canceled and not canceled bookings, essential for understanding the baseline cancellation rate.
- Lead Time: Analyzing the distribution of lead times (time between booking and stay) might reveal trends in cancellations related to booking anticipation.
- Stay Duration: The total nights stayed (weekend + weeknights) could indicate if longer or shorter stays are more prone to cancellation.
- Guest Type: The proportion of repeated guests versus new guests can highlight loyalty and its impact on cancellations.

**Bivariate Analysis**

- Lead Time vs. Cancellation: To see if longer lead times correlate with higher cancellation rates.
- Room Type vs. Cancellation: Understanding if specific room types are more - subject to cancellations.
- Meal Plan vs. Cancellation: To investigate if the choice of meal plan affects cancellation likelihood.
- Special Requests vs. Cancellation: Analyzing the relationship between the number of special requests and cancellations.

**Visualizations**

- Bar charts for categorical variables (meal plan, room type) against booking status.
- Histograms for numerical variables (lead time, stay duration) to understand distributions.
- Scatter plots or box plots for bivariate analysis to identify patterns and outliers.

**Key Observations**

- Potential findings could include a higher cancellation rate for longer lead times, a specific room type being more prone to cancellations, or repeated guests showing lower cancellation rates.

Data Pre-processing

- Missing Values: Impute or remove missing values based on their proportion and importance.
- Outliers: Identify and treat outliers in variables like lead time and stay duration to prevent model skew.
- Feature Engineering: Create new features such as total stay length or categorize lead times into bins for more nuanced analysis.
- Data Split: The data was split 70:30 for training and testing, ensuring a representative sample for model validation.


Model Building - Logistic Regression

- The logistic regression model indicated significant predictors like lead time, special requests, and room type, with their coefficients providing insights into their impact on cancellation likelihood.
- The model's assumptions, such as linearity in the log-odds and lack of multicollinearity, were addressed during pre-processing and feature selection.

Model Improvement - Logistic Regression

- Adjusting the classification threshold based on ROC curve analysis improved model sensitivity and specificity, better aligning with business objectives by possibly reducing false negatives (undetected cancellations).

Model Building - Decision Tree

- The decision tree model offered a different approach, providing a clear set of decision rules and highlighting the most influential factors leading to cancellations.

- Initial performance indicated areas for improvement, primarily in reducing overfitting and complexity.

Model Improvement - Decision Tree

- Pruning the decision tree using cost complexity pruning (ccp_alpha) improved the model by reducing complexity and overfitting, making it more generalizable.
The decision rules and feature importance derived from the pruned tree offered actionable insights, with lead time and special requests again being significant.

Pruning the Tree
- Pruning was necessary and beneficial for the decision tree model to prevent overfitting and improve its generalizability to unseen data.

**Actionable Insights & Recommendations**

- Priority Monitoring for Long Lead Times: Bookings made well in advance should be monitored more closely for potential cancellations.
- Special Request Fulfillment: Ensuring special requests are met could reduce cancellations, indicating the importance of personalized guest experiences.
- Loyalty Programs: Encouraging repeat bookings through loyalty programs might reduce cancellation rates, as repeated guests are less likely to cancel.
- Flexible Policies for High-Risk Bookings: Implementing flexible cancellation policies for identified high-risk bookings might preempt last-minute cancellations, allowing the hotel to manage inventory better.

Profitable Policies for Cancellations and Refunds:
1. Dynamic Cancellation Fees: Implementing a sliding scale for cancellation fees based on the lead time before the stay can incentivize guests to either keep their booking or cancel well in advance, allowing the hotel more time to rebook the room. For instance, cancellations made 30 days in advance might incur no fee, whereas cancellations within 48 hours of the stay could be charged a higher fee.

2. Non-Refundable Rates: Offering non-refundable rates as a cheaper option for guests who are certain about their travel plans can secure revenue upfront. This approach is particularly effective during peak seasons or special events when demand is high.

3. Deposit Requirement: For certain high-demand periods or for larger bookings, requiring a deposit that partially covers the stay can reduce cancellations. This deposit could be applied to the final bill or forfeited in case of a cancellation, based on the timing.

4. Rescheduling Options: Providing guests with the flexibility to reschedule their stay without a penalty can retain revenue that would otherwise be lost through cancellations. This policy should be promoted, especially in situations where cancellations are due to unforeseen circumstances.

5. Last-Minute Deals: For rooms that do become available due to cancellations, offering last-minute deals can help recoup lost revenue. These deals can be targeted to loyalty program members or marketed through platforms specializing in last-minute bookings.

 **Additional Recommendations:**
1. Overbooking Strategy: Implement a data-driven overbooking strategy that accounts for the historical rate of no-shows and cancellations. This can help optimize occupancy rates, but it needs to be managed carefully to avoid negative guest experiences.

2. Personalized Communication: Engage with guests who have booked far in advance at regular intervals leading up to their stay. This could involve personalized emails offering additional services or information about the property and the surrounding area, which could reduce the likelihood of cancellation.

3. Data Analysis for Targeted Marketing: Use data analytics to identify patterns in cancellations and target specific demographics with tailored marketing campaigns. For example, if data shows a high cancellation rate from a particular geographic region or booking platform, targeted offers or communication could be developed to address this.

4. Leverage Technology for Guest Engagement: Utilize mobile apps or SMS to engage with guests from the moment they book. Features could include countdowns to their stay, "know before you go" tips, and easy access to concierge services, making guests less likely to cancel.

5. Incentivize Direct Bookings: Encourage guests to book directly with the hotel through incentives such as free upgrades, complimentary breakfast, or loyalty points. Direct bookings often have lower cancellation rates and allow for more direct communication with guests.

**Conclusion:**
The combination of predictive modeling to identify potential cancellations and the strategic implementation of policies can significantly mitigate the impact of cancellations on INN Hotels. By understanding the key drivers behind cancellations through logistic regression and decision tree models, the hotel can tailor its approach to managing and reducing cancellations effectively. Adopting a mix of flexible and strict policies, depending on the booking's risk profile, can help balance guest satisfaction with revenue optimization. Implementing these data-driven strategies will empower INN Hotels to enhance operational efficiency, improve guest experiences, and ultimately, increase profitability.
